In [1]:
import rdflib
import requests
import json
import pandas as pd
from tqdm.auto import tqdm

/Users/gaignard-a/miniconda3/envs/abromics-graph/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#!wget 'https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJEB62462&result=read_run&fields=study_accession,sample_accession,experiment_accession,run_accession,tax_id,scientific_name,fastq_ftp,submitted_ftp,sra_ftp,bam_ftp&format=tsv&download=true&limit=0' -O ena_PRJEB62462.tsv

In [3]:
df = pd.read_csv("ena_PRJEB62462.tsv", sep="\t")
samples = df["sample_accession"].unique()
samples

<StringArray>
['SAMEA114388891', 'SAMEA114388888', 'SAMEA114388883', 'SAMEA114388896',
 'SAMEA114388884', 'SAMEA114388894', 'SAMEA114388889', 'SAMEA114388882',
 'SAMEA114388886', 'SAMEA114388897', 'SAMEA114388881', 'SAMEA114388895',
 'SAMEA114388892', 'SAMEA114388898', 'SAMEA114388885', 'SAMEA114388887',
 'SAMEA114388880', 'SAMEA114388893', 'SAMEA114388890']
Length: 19, dtype: str

In [18]:
for s in tqdm(samples):
    s_url = f"https://www.ebi.ac.uk/biosamples/samples/{s}.ldjson"
    response = requests.get(s_url)
    if response.status_code == 200:
        sample_data = response.json()
        # write sample data to a JSON file
        with open(f"samples_metadata/{s}.jsonld", "w") as f:
            json.dump(sample_data, f, indent=2)



  0%|          | 0/19 [00:00<?, ?it/s]

  5%|▌         | 1/19 [00:00<00:03,  4.88it/s]

 11%|█         | 2/19 [00:00<00:02,  6.62it/s]

 16%|█▌        | 3/19 [00:00<00:02,  7.77it/s]

 21%|██        | 4/19 [00:00<00:01,  8.49it/s]

 26%|██▋       | 5/19 [00:00<00:02,  6.98it/s]

 32%|███▏      | 6/19 [00:00<00:01,  7.53it/s]

 37%|███▋      | 7/19 [00:00<00:01,  8.12it/s]

 42%|████▏     | 8/19 [00:01<00:01,  8.44it/s]

 47%|████▋     | 9/19 [00:01<00:01,  7.25it/s]

 53%|█████▎    | 10/19 [00:01<00:01,  7.84it/s]

 58%|█████▊    | 11/19 [00:01<00:00,  8.31it/s]

 63%|██████▎   | 12/19 [00:01<00:00,  8.64it/s]

 68%|██████▊   | 13/19 [00:01<00:00,  7.14it/s]

 74%|███████▎  | 14/19 [00:01<00:00,  7.80it/s]

 79%|███████▉  | 15/19 [00:01<00:00,  8.23it/s]

 84%|████████▍ | 16/19 [00:02<00:00,  8.49it/s]

 89%|████████▉ | 17/19 [00:02<00:00,  7.10it/s]

 95%|█████████▍| 18/19 [00:02<00:00,  7.03it/s]

100%|██████████| 19/19 [00:02<00:00,  7.62it/s]


In [53]:
# iterate with glob over all JSON-LD files and load them into an RDF graph

prefixes = """
PREFIX abromics: <http://abromics.org/ontology#>
PREFIX sio: <http://semanticscience.org/resource/>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX geof:   <http://www.opengis.net/def/function/geosparql/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX sf:     <http://www.opengis.net/ont/sf#>
PREFIX geos:   <http://www.opengis.net/def/geosparql/>
PREFIX xsd:    <http://www.w3.org/2001/XMLSchema#>
PREFIX sosa:   <http://www.w3.org/ns/sosa/>
PREFIX schema: <http://schema.org/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX biosample: <http://identifiers.org/biosample/>
"""

construct_query = """
CONSTRUCT {
    ?sample rdf:type sio:001050, prov:Entity ;
        schema:title  ?title ;
        schema:identifier ?id ;
        prov:generatedAtTime ?date ;
        sio:000068 [
            a sio:001109 ;
            rdfs:label ?host ;
        ] ;

        sio:000068 [
            a sio:001109 ;
            rdfs:label ?host ;
        ] ;

        obo:RO_0002162 [ # in taxon 
            a obo:COB_0000022 ; #  organism
            rdfs:label ?organism ;
        ] ;

        prov:atLocation [
                a prov:Location , geo:Feature ;
                rdfs:label ?location ;
                rdfs:comment ?location_region ;
                geo:hasGeometry [
                    a geo:Geometry ;
                    geo:asWKT ?geometry ;
                ] ;
        ] . 
}
WHERE {
    ?rec a schema:DataRecord ;
        schema:mainEntity ?sample .

    OPTIONAL {
        ?sample schema:identifier ?id .
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "title" ;
            schema:value ?title ;
        ] .
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "collection date" ;
            schema:value ?date ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "host scientific name" ;
            schema:value ?host ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "isolation_source" ;
            schema:value ?source ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "organism" ;
            schema:value ?organism ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "geographic location (country and/or sea)" ;
            schema:value ?location ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "geographic location (region and locality)" ;
            schema:value ?location_region ;
        ] ;
    }

    OPTIONAL {
        ?sample schema:additionalProperty [
            schema:name "longitude" ;
            schema:value ?long ;
        ] .
        ?sample schema:additionalProperty [
            schema:name "latitude" ;
            schema:value ?lat ;
        ] .
        BIND(CONCAT("POINT(", STR(?long), " ", STR(?lat), ")") AS ?wktStr)
        BIND(STRDT(?wktStr, geo:wktLiteral) AS ?geometry)
    }
}
"""

In [ ]:
from rdflib.plugins.sparql.parser import parseQuery

G2 = rdflib.Graph()

for s in samples:
    g = rdflib.Graph()
    # bind namespaces
    
    g.parse(f"samples_metadata/{s}.jsonld", format="json-ld")

    q = prefixes+construct_query
    #print(q)
    parseQuery(q)  # check if the query is valid

    g_constructed = g.query(q)

    G2 += g_constructed

print(G2.serialize(format="turtle"))
G2.serialize("abromics_aligned_samples.ttl", format="turtle")
#print(f"Loaded metadata for {len(samples)} samples into RDF graph with {len(g)} triples")
#print(g.serialize(format="turtle"))

@prefix geo: <http://www.opengis.net/ont/geosparql#> .
@prefix ns1: <http://schema.org/> .
@prefix ns2: <http://purl.obolibrary.org/obo/> .
@prefix ns3: <http://semanticscience.org/resource/> .
@prefix prov: <http://www.w3.org/ns/prov#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<https://www.ebi.ac.uk/biosamples/samples/SAMEA114388880> a ns3:001050,
        prov:Entity ;
    ns2:RO_0002162 [ a ns2:COB_0000022 ;
            rdfs:label "Escherichia coli" ] ;
    ns1:identifier "biosample:SAMEA114388880" ;
    ns1:title "56048_A71C_Chicken_Backyard" ;
    ns3:000068 [ a ns3:001109 ;
            rdfs:label "Gallus gallus" ],
        [ a ns3:001109 ;
            rdfs:label "Gallus gallus" ] ;
    prov:atLocation [ a geo:Feature,
                prov:Location ;
            rdfs:label "Chile" ;
            geo:hasGeometry [ a geo:Geometry ] ;
            rdfs:comment "Chacabuco Province" ] ;
    prov:generatedAtTime "2019" .

<https://www.ebi.ac.uk/biosamples/samples/SAMEA1143